# STW Full-History Explorer

This notebook is the interactive viewer for the full STW mV/Irr dataset.

- Each metric is shown as a simple full-history line plot.
- Missing values and cleaned outliers remain `NaN`, which creates visible gaps without dropping timestamps.
- The notebook renders full-history views for `GHI`, `DHI`, and `DNI` across the complete dataset time range.
- Use `plot_metric_window(...)` only when you need exact values in a smaller time window.


In [ ]:
from pathlib import Path

import holoviews as hv
import hvplot.pandas
import pandas as pd

hv.extension("bokeh")

ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "plots" else Path.cwd().resolve()
PARQUET_PATH = ROOT_DIR / 'final output/stw_mV_Irr.parquet'

df = pd.read_parquet(PARQUET_PATH)
df["datetime"] = pd.to_datetime(df["datetime"])
metrics = sorted(column[:-3] for column in df.columns if column.endswith("_mV"))


In [ ]:
def plot_metric(metric: str):
    mv_column = f"{metric}_mV"
    irr_column = f"{metric}_Irr"
    metric_df = df[["datetime", mv_column, irr_column]].copy()

    return metric_df.hvplot.line(
        x="datetime",
        y=[mv_column, irr_column],
        responsive=True,
        min_height=520,
        xlabel="Time",
        ylabel="Value",
        title=f"{metric} full history",
        legend="top",
    )


def plot_metric_window(metric: str, start: str, end: str):
    mv_column = f"{metric}_mV"
    irr_column = f"{metric}_Irr"
    window = df.loc[
        (df["datetime"] >= pd.Timestamp(start)) & (df["datetime"] <= pd.Timestamp(end)),
        ["datetime", mv_column, irr_column],
    ].copy()
    if window.empty:
        raise ValueError(f"No rows found for {metric} between {start} and {end}.")

    return window.hvplot.line(
        x="datetime",
        y=[mv_column, irr_column],
        responsive=True,
        min_height=520,
        xlabel="Time",
        ylabel="Value",
        title=f"{metric} detailed window: {start} to {end}",
        legend="top",
    )


## GHI Full History


In [ ]:
plot_metric("GHI")

## DHI Full History


In [ ]:
plot_metric("DHI")

## DNI Full History


In [ ]:
plot_metric("DNI")

In [ ]:
# Optional exact-window helper example:
# plot_metric_window("GHI", "2024-02-01 00:00", "2024-02-03 00:00")